# ♻️ AI Agent for Garbage Classification and Smart Recycling Recommendation

**Final-year B.Tech project — end-to-end notebook**

This notebook takes you from raw Kaggle data to a trained, compared, and
explainable garbage-classification model with a recycling recommendation
engine, following this order (do not skip ahead to the app):

1. Setup → 2. Dataset → 3. EDA & cleaning → 4. Split & augmentation →
5. Train EfficientNet-B3 / ResNet50 / MobileNetV3 → 6. Evaluate & compare →
7. Select best model → 8. Recycling recommendations → 9. Grad-CAM →
10. Streamlit app → 11. Final report.

**Runtime:** Colab → *Runtime → Change runtime type → GPU (T4)*.

All dataset statistics and model metrics in this notebook are computed
live from your actual run — nothing is pre-filled or fabricated. The first
time you run this end-to-end, expect full training (all 3 models) to take
roughly 30–90 minutes on a Colab T4, depending on dataset size and epochs.

## CELL 1 — Install Dependencies

In [ ]:
!pip install -q kaggle torchmetrics
# torch/torchvision/numpy/pandas/matplotlib/seaborn/scikit-learn/Pillow/opencv/tqdm
# are already preinstalled in Colab; we just confirm versions below.

## CELL 2 — Clone/Mount Project Code

If you're running this notebook standalone in Colab (not from the cloned repo), mount Drive and/or clone your repo so `src/` is importable. If you uploaded the whole `garbage-classification-ai/` folder to Colab's file system already, skip straight to CELL 3.

In [ ]:
import os, sys

PROJECT_ROOT = "/content/garbage-classification-ai"

# Option A: you already uploaded/cloned the project folder to /content
if os.path.exists(PROJECT_ROOT):
    print(f"Found project at {PROJECT_ROOT}")
else:
    # Option B: clone from your own GitHub repo (replace URL after you push it)
    # !git clone https://github.com/<your-username>/garbage-classification-ai.git /content/garbage-classification-ai
    print("Project folder not found — upload it via the Colab file browser, "
          "or uncomment the git clone line above once you've pushed this repo to GitHub.")

sys.path.append(PROJECT_ROOT)

## CELL 3 — Imports

In [ ]:
import os
import time
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src import config, utils, dataset, preprocessing, models, train, evaluate, recycling, gradcam

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", config.DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## CELL 4 — Configuration

All hyperparameters live in `src/config.py` (single source of truth — see Section 14 of the spec). Print them here so they're visible in the run log for reproducibility.

In [ ]:
utils.set_seed(config.SEED)

print("SEED:", config.SEED)
print("IMAGE_SIZE:", config.IMAGE_SIZE)
print("BATCH_SIZE:", config.BATCH_SIZE)
print("NUM_EPOCHS_HEAD:", config.NUM_EPOCHS_HEAD)
print("NUM_EPOCHS_FINETUNE:", config.NUM_EPOCHS_FINETUNE)
print("LEARNING_RATE_HEAD:", config.LEARNING_RATE_HEAD)
print("LEARNING_RATE_FINETUNE:", config.LEARNING_RATE_FINETUNE)
print("TRAIN/VAL/TEST split:", config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO)
print("CONFIDENCE_THRESHOLD:", config.CONFIDENCE_THRESHOLD)

## CELL 5 — Dataset Download

### Method 1 — Kaggle API (recommended)
1. Go to kaggle.com → your profile → **Settings** → **API** → **Create New Token**. This downloads `kaggle.json` to your computer.
2. Run the cell below, and when prompted, upload `kaggle.json`.

### Method 2 — Manual download
If you'd rather not use the API: download the zip directly from https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification , then use the Colab file browser (left sidebar → folder icon → upload) to upload the zip into `data/raw/`, and run:
`!unzip -q data/raw/<the-file>.zip -d data/raw/`
then skip the Kaggle-API cell below.

In [ ]:
from google.colab import files

KAGGLE_JSON_PATH = "/root/.kaggle/kaggle.json"

if not os.path.exists(KAGGLE_JSON_PATH):
    print("Upload your kaggle.json now...")
    uploaded = files.upload()  # select kaggle.json from your machine
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        os.system(f"cp {fname} {KAGGLE_JSON_PATH}")
    os.system(f"chmod 600 {KAGGLE_JSON_PATH}")
    print("kaggle.json installed.")
else:
    print("kaggle.json already present.")

In [ ]:
# Download + unzip straight into data/raw (Method 1)
os.makedirs(config.RAW_DATA_DIR, exist_ok=True)
!kaggle datasets download -d {config.DATASET_KAGGLE_SLUG} -p {config.RAW_DATA_DIR} --unzip
print("Download complete. Contents of data/raw:")
!ls {config.RAW_DATA_DIR}

## CELL 6 — Dataset Structure Explanation

The Kaggle archive for this dataset has historically shipped as nested folders like `Garbage classification/Garbage classification/<class>/*.jpg`, though this can vary between mirrors/versions. Rather than hard-coding a path, `dataset.discover_class_folders()` walks the whole raw-data tree and treats any folder that directly contains image files as a class — this keeps the notebook working even if the extracted layout differs.

In [ ]:
class_to_images = dataset.discover_class_folders(config.RAW_DATA_DIR)
print(f"Discovered {len(class_to_images)} class folders:")
for cls, paths in sorted(class_to_images.items()):
    print(f"  {cls}: {len(paths)} images")

## CELL 7 — Dataset Exploration (EDA)

Compute the real numbers requested in Section 1: class count, class names, per-class counts, total images, min/max/average per class. **Nothing here is hard-coded** — it's all derived from `class_to_images` above.

In [ ]:
summary = dataset.dataset_summary(class_to_images)

print("Number of classes:", summary["num_classes"])
print("Class names:", summary["class_names"])
print("Total images:", summary["total_images"])
print("Min images per class:", summary["min_images"])
print("Max images per class:", summary["max_images"])
print(f"Average images per class: {summary['avg_images']:.1f}")

pd.DataFrame(list(summary["counts_per_class"].items()), columns=["class", "count"]) \
  .sort_values("count", ascending=False)

## CELL 8 — Dataset Cleaning (corrupted / duplicate / tiny images)

`dataset.audit_dataset()` opens every single image to check it's readable, hashes it to catch exact duplicates, and flags images smaller than 32px on a side. `clean_dataset()` then returns a filtered dict with flagged files removed — nothing is deleted from disk, only excluded from the in-memory index used for training.

In [ ]:
report = dataset.audit_dataset(class_to_images)

print("Total images scanned:", report["total_scanned"])
print("Corrupted images found:", len(report["corrupted"]))
print("Duplicate images found:", len(report["duplicates"]))
print("Too-small images found:", len(report["too_small"]))

if report["corrupted"]:
    print("\nExample corrupted paths:", report["corrupted"][:3])
if report["duplicates"]:
    print("Example duplicate pairs:", report["duplicates"][:3])

clean_class_to_images = dataset.clean_dataset(class_to_images, report)
clean_summary = dataset.dataset_summary(clean_class_to_images)
print("\nTotal images after cleaning:", clean_summary["total_images"])

## CELL 9 — Class Distribution Visualization

In [ ]:
counts_df = pd.DataFrame(list(clean_summary["counts_per_class"].items()), columns=["class", "count"]) \
              .sort_values("count", ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(data=counts_df, x="class", y="count", palette="viridis")
plt.title("Class Distribution (post-cleaning)")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "class_distribution.png"), dpi=150)
plt.show()

imbalance_ratio = clean_summary["max_images"] / clean_summary["min_images"]
print(f"Max/Min class-size ratio: {imbalance_ratio:.2f}x")
if imbalance_ratio > 1.5:
    print("-> Non-trivial class imbalance detected. We handle this in CELL 12 "
          "with a WeightedRandomSampler + class-weighted loss.")
else:
    print("-> Classes are reasonably balanced.")

## CELL 10 — Sample Images per Class

In [ ]:
class_names_sorted = clean_summary["class_names"]
n_classes = len(class_names_sorted)
n_samples = 4

fig, axes = plt.subplots(n_classes, n_samples, figsize=(n_samples * 2.2, n_classes * 2.2))
for row, cls in enumerate(class_names_sorted):
    sample_paths = random.sample(clean_class_to_images[cls], min(n_samples, len(clean_class_to_images[cls])))
    for col in range(n_samples):
        ax = axes[row, col] if n_classes > 1 else axes[col]
        if col < len(sample_paths):
            img = Image.open(sample_paths[col]).convert("RGB")
            ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls, fontsize=10)
    axes[row, 0].set_title(cls, fontsize=11, loc="left") if n_classes > 1 else None

plt.suptitle("Sample Images per Class", y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "sample_images_per_class.png"), dpi=150)
plt.show()

## CELL 11 — Train / Validation / Test Split

70/15/15 stratified split per class, fixed seed for reproducibility. Because splitting happens on the file-path list (not on augmented copies), the same image can never land in two splits — this avoids the data leakage warned about in Section 6.

In [ ]:
train_items, val_items, test_items, class_names = dataset.stratified_split(clean_class_to_images, seed=config.SEED)

print(f"Classes ({len(class_names)}):", class_names)
print(f"Train images: {len(train_items)}")
print(f"Val images:   {len(val_items)}")
print(f"Test images:  {len(test_items)}")

# Sanity check: no path appears in more than one split
train_paths = {p for p, _ in train_items}
val_paths = {p for p, _ in val_items}
test_paths = {p for p, _ in test_items}
assert not (train_paths & val_paths), "Leakage between train and val!"
assert not (train_paths & test_paths), "Leakage between train and test!"
assert not (val_paths & test_paths), "Leakage between val and test!"
print("No overlap between splits confirmed.")

## CELL 12 — Data Augmentation & DataLoaders

Training transforms include random crop/flip/rotation/color-jitter/affine; validation and test transforms are deterministic (resize + center-crop only), per Section 5. Class imbalance (if flagged in CELL 9) is handled via a `WeightedRandomSampler` on the training loader plus class-weighted loss in the training cells below.

In [ ]:
train_loader, val_loader, test_loader = preprocessing.build_dataloaders(
    train_items, val_items, test_items, class_names,
    batch_size=config.BATCH_SIZE, use_weighted_sampler=True,
)

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)  # (batch, 3, H, W)
print("Label sample:", labels[:8].tolist())

### Augmentation Preview (Section 31)

Compare an original image against several augmented versions.

In [ ]:
sample_path = train_items[0][0]
original = Image.open(sample_path).convert("RGB")
train_transform = preprocessing.get_train_transform()

fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
axes[0].imshow(original); axes[0].set_title("Original"); axes[0].axis("off")
for i in range(1, 5):
    aug_tensor = train_transform(original)
    aug_img = aug_tensor.permute(1, 2, 0).numpy()
    aug_img = np.clip(aug_img * np.array(config.IMAGENET_STD) + np.array(config.IMAGENET_MEAN), 0, 1)
    axes[i].imshow(aug_img); axes[i].set_title(f"Augmented {i}"); axes[i].axis("off")
plt.suptitle("Augmentation is applied only to TRAINING images — it teaches the\n"
             "model to be invariant to lighting/angle/crop, which garbage photos\n"
             "in the wild vary a lot on.")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "augmentation_preview.png"), dpi=150)
plt.show()

In [ ]:
results = {}

## CELL 14 — EfficientNet-B3

Build `efficientnet_b3` with an ImageNet-pretrained backbone (frozen) and a fresh classifier head sized to `len(class_names)`.

In [ ]:
model_efficientnet_b3 = models.build_model("efficientnet_b3", num_classes=len(class_names), freeze_backbone=True)
model_efficientnet_b3.to(config.DEVICE)
print(f"{'efficientnet_b3'} parameters (trainable): {utils.count_parameters(model_efficientnet_b3):,}")

## CELL 15 — Train EfficientNet-B3 (Phase 1: head-only, then Phase 2: fine-tune)

In [ ]:
class_weights = dataset.compute_class_weights(train_items, len(class_names))
criterion = train.make_criterion(class_weights, device=config.DEVICE)

# Phase 1: train the classifier head with the backbone frozen
optimizer = train.make_optimizer(model_efficientnet_b3, lr=config.LEARNING_RATE_HEAD)
scheduler = train.make_scheduler(optimizer)

start_time = time.time()
model_efficientnet_b3, history_efficientnet_b3_head, best_val_acc = train.train_model(
    model_efficientnet_b3, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=config.NUM_EPOCHS_HEAD, device=config.DEVICE,
    model_name="efficientnet_b3", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["efficientnet_b3"],
    class_names=class_names,
)

# Phase 2: unfreeze the last couple of backbone blocks and fine-tune at a lower LR
model_efficientnet_b3 = models.unfreeze_for_finetuning(model_efficientnet_b3, "efficientnet_b3", num_blocks=2)
optimizer_ft = train.make_optimizer(model_efficientnet_b3, lr=config.LEARNING_RATE_FINETUNE)
scheduler_ft = train.make_scheduler(optimizer_ft)

model_efficientnet_b3, history_efficientnet_b3_ft, best_val_acc = train.train_model(
    model_efficientnet_b3, train_loader, val_loader, criterion, optimizer_ft, scheduler_ft,
    epochs=config.NUM_EPOCHS_FINETUNE, device=config.DEVICE,
    model_name="efficientnet_b3", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["efficientnet_b3"],
    class_names=class_names,
)

training_time_efficientnet_b3 = time.time() - start_time
print(f"Total training time for efficientnet_b3: {training_time_efficientnet_b3:.1f}s")

# Merge phase-1 + phase-2 history for the training-curve plot
history_efficientnet_b3 = {
    k: history_efficientnet_b3_head[k] + history_efficientnet_b3_ft[k] for k in history_efficientnet_b3_head
}
utils.plot_training_curves(history_efficientnet_b3, "efficientnet_b3")

## CELL 16 — Evaluate EfficientNet-B3

In [ ]:
y_pred_efficientnet_b3, y_true_efficientnet_b3, y_prob_efficientnet_b3 = evaluate.get_predictions(model_efficientnet_b3, test_loader)
metrics_efficientnet_b3 = evaluate.compute_metrics(y_true_efficientnet_b3, y_pred_efficientnet_b3, class_names)

print(f"EfficientNet-B3 — Test Accuracy: {metrics_efficientnet_b3['accuracy']*100:.2f}%")
print(f"Precision: {metrics_efficientnet_b3['precision']:.4f}  "
      f"Recall: {metrics_efficientnet_b3['recall']:.4f}  "
      f"F1: {metrics_efficientnet_b3['f1']:.4f}  "
      f"Macro-F1: {metrics_efficientnet_b3['macro_f1']:.4f}")

utils.plot_confusion_matrix(metrics_efficientnet_b3['confusion_matrix'], class_names, "efficientnet_b3")

confused_efficientnet_b3 = evaluate.most_confused_pairs(metrics_efficientnet_b3['confusion_matrix'], class_names)
print("\nMost confused class pairs (true -> predicted, count):")
for true_c, pred_c, cnt in confused_efficientnet_b3:
    print(f"  {true_c} -> {pred_c}: {cnt}")

results["efficientnet_b3"] = {
    "model": model_efficientnet_b3,
    "metrics": metrics_efficientnet_b3,
    "training_time_sec": training_time_efficientnet_b3,
    "image_size": config.IMAGE_SIZE,
}

## CELL 17 — ResNet50

Build `resnet50` with an ImageNet-pretrained backbone (frozen) and a fresh classifier head sized to `len(class_names)`.

In [ ]:
model_resnet50 = models.build_model("resnet50", num_classes=len(class_names), freeze_backbone=True)
model_resnet50.to(config.DEVICE)
print(f"{'resnet50'} parameters (trainable): {utils.count_parameters(model_resnet50):,}")

## CELL 18 — Train ResNet50 (Phase 1: head-only, then Phase 2: fine-tune)

In [ ]:
class_weights = dataset.compute_class_weights(train_items, len(class_names))
criterion = train.make_criterion(class_weights, device=config.DEVICE)

# Phase 1: train the classifier head with the backbone frozen
optimizer = train.make_optimizer(model_resnet50, lr=config.LEARNING_RATE_HEAD)
scheduler = train.make_scheduler(optimizer)

start_time = time.time()
model_resnet50, history_resnet50_head, best_val_acc = train.train_model(
    model_resnet50, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=config.NUM_EPOCHS_HEAD, device=config.DEVICE,
    model_name="resnet50", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["resnet50"],
    class_names=class_names,
)

# Phase 2: unfreeze the last couple of backbone blocks and fine-tune at a lower LR
model_resnet50 = models.unfreeze_for_finetuning(model_resnet50, "resnet50", num_blocks=2)
optimizer_ft = train.make_optimizer(model_resnet50, lr=config.LEARNING_RATE_FINETUNE)
scheduler_ft = train.make_scheduler(optimizer_ft)

model_resnet50, history_resnet50_ft, best_val_acc = train.train_model(
    model_resnet50, train_loader, val_loader, criterion, optimizer_ft, scheduler_ft,
    epochs=config.NUM_EPOCHS_FINETUNE, device=config.DEVICE,
    model_name="resnet50", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["resnet50"],
    class_names=class_names,
)

training_time_resnet50 = time.time() - start_time
print(f"Total training time for resnet50: {training_time_resnet50:.1f}s")

# Merge phase-1 + phase-2 history for the training-curve plot
history_resnet50 = {
    k: history_resnet50_head[k] + history_resnet50_ft[k] for k in history_resnet50_head
}
utils.plot_training_curves(history_resnet50, "resnet50")

## CELL 19 — Evaluate ResNet50

In [ ]:
y_pred_resnet50, y_true_resnet50, y_prob_resnet50 = evaluate.get_predictions(model_resnet50, test_loader)
metrics_resnet50 = evaluate.compute_metrics(y_true_resnet50, y_pred_resnet50, class_names)

print(f"ResNet50 — Test Accuracy: {metrics_resnet50['accuracy']*100:.2f}%")
print(f"Precision: {metrics_resnet50['precision']:.4f}  "
      f"Recall: {metrics_resnet50['recall']:.4f}  "
      f"F1: {metrics_resnet50['f1']:.4f}  "
      f"Macro-F1: {metrics_resnet50['macro_f1']:.4f}")

utils.plot_confusion_matrix(metrics_resnet50['confusion_matrix'], class_names, "resnet50")

confused_resnet50 = evaluate.most_confused_pairs(metrics_resnet50['confusion_matrix'], class_names)
print("\nMost confused class pairs (true -> predicted, count):")
for true_c, pred_c, cnt in confused_resnet50:
    print(f"  {true_c} -> {pred_c}: {cnt}")

results["resnet50"] = {
    "model": model_resnet50,
    "metrics": metrics_resnet50,
    "training_time_sec": training_time_resnet50,
    "image_size": config.IMAGE_SIZE,
}

## CELL 20 — MobileNetV3

Build `mobilenet_v3` with an ImageNet-pretrained backbone (frozen) and a fresh classifier head sized to `len(class_names)`.

In [ ]:
model_mobilenet_v3 = models.build_model("mobilenet_v3", num_classes=len(class_names), freeze_backbone=True)
model_mobilenet_v3.to(config.DEVICE)
print(f"{'mobilenet_v3'} parameters (trainable): {utils.count_parameters(model_mobilenet_v3):,}")

## CELL 21 — Train MobileNetV3 (Phase 1: head-only, then Phase 2: fine-tune)

In [ ]:
class_weights = dataset.compute_class_weights(train_items, len(class_names))
criterion = train.make_criterion(class_weights, device=config.DEVICE)

# Phase 1: train the classifier head with the backbone frozen
optimizer = train.make_optimizer(model_mobilenet_v3, lr=config.LEARNING_RATE_HEAD)
scheduler = train.make_scheduler(optimizer)

start_time = time.time()
model_mobilenet_v3, history_mobilenet_v3_head, best_val_acc = train.train_model(
    model_mobilenet_v3, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=config.NUM_EPOCHS_HEAD, device=config.DEVICE,
    model_name="mobilenet_v3", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["mobilenet_v3"],
    class_names=class_names,
)

# Phase 2: unfreeze the last couple of backbone blocks and fine-tune at a lower LR
model_mobilenet_v3 = models.unfreeze_for_finetuning(model_mobilenet_v3, "mobilenet_v3", num_blocks=2)
optimizer_ft = train.make_optimizer(model_mobilenet_v3, lr=config.LEARNING_RATE_FINETUNE)
scheduler_ft = train.make_scheduler(optimizer_ft)

model_mobilenet_v3, history_mobilenet_v3_ft, best_val_acc = train.train_model(
    model_mobilenet_v3, train_loader, val_loader, criterion, optimizer_ft, scheduler_ft,
    epochs=config.NUM_EPOCHS_FINETUNE, device=config.DEVICE,
    model_name="mobilenet_v3", checkpoint_path=config.MODEL_CHECKPOINT_PATHS["mobilenet_v3"],
    class_names=class_names,
)

training_time_mobilenet_v3 = time.time() - start_time
print(f"Total training time for mobilenet_v3: {training_time_mobilenet_v3:.1f}s")

# Merge phase-1 + phase-2 history for the training-curve plot
history_mobilenet_v3 = {
    k: history_mobilenet_v3_head[k] + history_mobilenet_v3_ft[k] for k in history_mobilenet_v3_head
}
utils.plot_training_curves(history_mobilenet_v3, "mobilenet_v3")

## CELL 22 — Evaluate MobileNetV3

In [ ]:
y_pred_mobilenet_v3, y_true_mobilenet_v3, y_prob_mobilenet_v3 = evaluate.get_predictions(model_mobilenet_v3, test_loader)
metrics_mobilenet_v3 = evaluate.compute_metrics(y_true_mobilenet_v3, y_pred_mobilenet_v3, class_names)

print(f"MobileNetV3 — Test Accuracy: {metrics_mobilenet_v3['accuracy']*100:.2f}%")
print(f"Precision: {metrics_mobilenet_v3['precision']:.4f}  "
      f"Recall: {metrics_mobilenet_v3['recall']:.4f}  "
      f"F1: {metrics_mobilenet_v3['f1']:.4f}  "
      f"Macro-F1: {metrics_mobilenet_v3['macro_f1']:.4f}")

utils.plot_confusion_matrix(metrics_mobilenet_v3['confusion_matrix'], class_names, "mobilenet_v3")

confused_mobilenet_v3 = evaluate.most_confused_pairs(metrics_mobilenet_v3['confusion_matrix'], class_names)
print("\nMost confused class pairs (true -> predicted, count):")
for true_c, pred_c, cnt in confused_mobilenet_v3:
    print(f"  {true_c} -> {pred_c}: {cnt}")

results["mobilenet_v3"] = {
    "model": model_mobilenet_v3,
    "metrics": metrics_mobilenet_v3,
    "training_time_sec": training_time_mobilenet_v3,
    "image_size": config.IMAGE_SIZE,
}

## CELL 23 — Compare All Models (Section 11)

A single fair comparison table: same data splits, same training loop, same evaluation code for all three architectures.

In [ ]:
comparison_df = evaluate.build_comparison_table(results)
comparison_df

## CELL 24 — Select Best Model (Section 33)

Selection uses a combined score of test F1 (70% weight) and inference speed (30% weight) — not simply 'biggest model wins'.

In [ ]:
best_info = evaluate.select_best_model(comparison_df, f1_weight=0.7, speed_weight=0.3)

print("=" * 40)
print("BEST MODEL")
print("=" * 40)
print(f"Model: {best_info['model_name']}")
print(f"Test Accuracy: {best_info['test_accuracy']*100:.2f}%")
print(f"F1 Score: {best_info['f1_score']:.4f}")
print(f"Inference Time: {best_info['inference_time_ms']:.2f} ms")
print(f"Reason: {best_info['reason']}")

utils.save_json(best_info, config.BEST_MODEL_INFO_PATH)

## CELL 25 — Save Best Model

Copy the winning model's checkpoint to `models/best_model.pth` — this is the exact file `src/predict.py` and the Streamlit app load by default.

In [ ]:
import shutil

best_model_name = best_info["model_name"]
src_ckpt = config.MODEL_CHECKPOINT_PATHS[best_model_name]
shutil.copyfile(src_ckpt, config.BEST_MODEL_PATH)
print(f"Copied {src_ckpt} -> {config.BEST_MODEL_PATH}")

## CELL 26 — Test Single Image Prediction (Section 17)

In [ ]:
from src.predict import GarbagePredictor

predictor = GarbagePredictor(config.BEST_MODEL_PATH)

# Swap in any test image path you like
sample_image_path = test_items[0][0]
result = predictor.predict_image(sample_image_path)

print("Prediction:", result["predicted_class"])
print(f"Confidence: {result['confidence']*100:.1f}%")
print("\nTop-3:")
for i, p in enumerate(result["top_predictions"], 1):
    print(f"{i}. {p['class']} - {p['confidence']*100:.1f}%")

plt.imshow(Image.open(sample_image_path))
plt.title(f"Predicted: {result['predicted_class']} ({result['confidence']*100:.1f}%)")
plt.axis("off")
plt.show()

## CELL 27 — Smart Recycling Recommendation (Section 19-20)

In [ ]:
from src.recycling import format_recommendation_text

if result["recycling_recommendation"] is not None:
    print(format_recommendation_text(
        result["predicted_class"], result["confidence"], result["recycling_recommendation"]
    ))
else:
    print(result["message"])

## CELL 28 — Grad-CAM Explainability (Section 30)

Grad-CAM is an **approximate** visual explanation of which image regions most influenced the prediction — it is not proof of the model's internal reasoning.

In [ ]:
from src.gradcam import GradCAM, overlay_heatmap

cam_tool = GradCAM(predictor.model, predictor.model_name)

image = Image.open(sample_image_path).convert("RGB")
tensor = predictor.transform(image).unsqueeze(0)
class_idx = predictor.class_names.index(result["predicted_class"])
cam, _ = cam_tool.generate(tensor, class_idx=class_idx)

resized = image.resize((predictor.image_size, predictor.image_size))
overlay = overlay_heatmap(np.array(resized), cam)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(resized); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(overlay); axes[1].set_title("Grad-CAM Heatmap"); axes[1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "gradcam_example.png"), dpi=150)
plt.show()

## CELL 29 — Run the Streamlit Application

Colab can't render a Streamlit server's UI directly, but you can tunnel it out. Simplest option: run this notebook's `models/` folder + repo locally (`streamlit run app/streamlit_app.py`) once you've trained and saved `best_model.pth`. If you want to preview from Colab itself:

In [ ]:
!pip install -q localtunnel
get_ipython().system_raw('streamlit run app/streamlit_app.py --server.port 8501 &')
import time; time.sleep(3)
!npx localtunnel --port 8501

## CELL 30 — Final Results Report (Section 34)

In [ ]:
print("=" * 42)
print("FINAL MODEL COMPARISON")
print("=" * 42)
for _, row in comparison_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"Accuracy: {row['Test Accuracy']*100:.2f}%")
    print(f"Precision: {row['Precision']*100:.2f}%")
    print(f"Recall: {row['Recall']*100:.2f}%")
    print(f"F1 Score: {row['F1 Score']*100:.2f}%")

print("\n" + "=" * 42)
print("BEST MODEL")
print("=" * 42)
print(f"Model: {best_info['model_name']}")
print(f"Test Accuracy: {best_info['test_accuracy']*100:.2f}%")
print(f"F1 Score: {best_info['f1_score']:.4f}")
print(f"Inference Time: {best_info['inference_time_ms']:.2f} ms")

## CELL 31 — Conclusion

Summarise, in your own words once you've seen the real numbers above:
- Which model won and by how much
- What the confusion matrix reveals about visually-similar classes (e.g. glass vs. plastic reflections, paper vs. cardboard texture)
- What you'd try next (more data for the weakest class, larger images, ensembling, longer fine-tuning, etc.)

See the project `README.md` for the full write-up template (abstract, objectives, methodology, advantages/limitations/future scope) to reuse for your report and viva.